In [1]:
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration


In [2]:
model=T5ForConditionalGeneration.from_pretrained("t5-small")
tokenizer=T5Tokenizer.from_pretrained("t5-small")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [3]:
article = """
NASA's Perseverance rover has discovered organic molecules on Mars,
suggesting the planet may have once harbored microbial life. Scientists
analyzed rock samples from the Jezero Crater, where an ancient river
delta existed billions of years ago. The findings have renewed interest
in future Mars missions aimed at returning samples to Earth for deeper
analysis. Researchers caution that organic molecules alone do not confirm
life, but represent a promising sign worth investigating further.
"""

In [4]:
input_ids=tokenizer.encode(

    "summarize:" + article,
    return_tensors="pt",
    max_length=512,
    truncation=True

)

In [5]:
output=model.generate(
    input_ids,
    max_length=100,
    min_length=30,
    num_beams=4,
    length_penalty=2.0,    
    early_stopping=True
)

In [6]:
summary = tokenizer.decode(output[0], skip_special_tokens=True)
print("Summary:", summary)

Summary: scientists analyzed rock samples from the Jezero Crater. findings renewed interest in future Mars missions aimed at returning samples to Earth.


In [7]:
from datasets import load_dataset

# Load a small slice to explore
dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:1%]")


# ---

In [8]:
print(dataset)
print(dataset[0].keys())


sample = dataset[0]
print("ARTICLE:\n", sample["article"][:500])
print("\nHIGHLIGHT (ground truth summary):\n", sample["highlights"])

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 2871
})
dict_keys(['article', 'highlights', 'id'])
ARTICLE:
 LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as s

HIGHLIGHT (ground truth summary):
 Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


In [9]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

generated = "NASA found organic molecules on Mars suggesting ancient life."
reference = "Perseverance rover discovered organic molecules on Mars near Jezero Crater."

scores = scorer.score(reference, generated)
for key, value in scores.items():
    print(f"{key}: Precision={value.precision:.3f}, Recall={value.recall:.3f}, F1={value.fmeasure:.3f}")

rouge1: Precision=0.444, Recall=0.400, F1=0.421
rouge2: Precision=0.375, Recall=0.333, F1=0.353
rougeL: Precision=0.444, Recall=0.400, F1=0.421


In [10]:

dataset = load_dataset("cnn_dailymail", "3.0.0")

# Preprocessing function
def preprocess(batch):
    inputs = ["summarize: " + doc for doc in batch["article"]]
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        batch["highlights"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply preprocessing
tokenized = dataset.map(preprocess, batched=True, remove_columns=dataset["train"].column_names)
print("Preprocessing done!", tokenized)

Preprocessing done! DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 11490
    })
})


In [11]:
from transformers import T5ForConditionalGeneration, T5Tokenizer

# No training needed — use pre-trained directly
model = T5ForConditionalGeneration.from_pretrained("t5-small")
tokenizer = T5Tokenizer.from_pretrained("t5-small")

def summarize(text, max_length=150, min_length=40):
    input_ids = tokenizer.encode(
        "summarize: " + text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )
    output = model.generate(
        input_ids,
        max_length=max_length,
        min_length=min_length,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Test it
article = """
NASA's Perseverance rover has discovered organic molecules on Mars,
suggesting the planet may have once harbored microbial life. Scientists
analyzed rock samples from the Jezero Crater, where an ancient river
delta existed billions of years ago.
"""
print(summarize(article))

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

scientists analyzed rock samples from the Jezero Crater, where an ancient river delta existed billions of years ago. the planet may have harbored microbial life, suggesting it harbored microbial life.
